In [1]:
import pandas as pd
import warnings
import os
warnings.filterwarnings("ignore")

In [2]:
def Read_data():
    df = pd.read_csv(f'{os.getcwd()}/result/ECERIECSIlegales(EdadInicio).csv')
    return df

def dict_impactosust():
    return {
        1 : 'Tabaco',
        2 : 'Alcohol',
        3 : 'Marihuana',
        4 : 'Hachis',
        5 : 'Cocaina',
        6 : 'Crack',
        7 : 'Otras Presentaciones (Basuco o pasta base, cocaina negra)',
        8 : 'Solventes y removedores',
        9 : 'Pegamento',
        10 : 'Esmaltes y pinturas',
        11 : 'Otros (aire comprimido, gasolinas y combustibles)',
        12 : 'Anfetaminas',
        13 : 'Metanfetaminas',
        14 : 'MDMA(extasis) y metanfetaminas alucinogenas (DMT)',
        15 : 'Otros (derivados anfetaminicos)',
        16 : 'LSD',
        17 : 'Plantas alucinogenas y derivados',
        18 : 'Otras (PCP, ketamina, excepto metanfetamina)',
        19 : 'Benzodiazepinas',
        20 : 'Rohypnol',
        21 : 'Otras (sedantes hiptnoticos, GHB)',
        22 : 'Heroina',
        23 : 'Opiaceos sinteticos (propoxifeno, nailbufina)',
        24 : 'Opio y opiodes (morfina, codeina)',
        25 : 'Con utilidad medica (Prozac, Paxil, Carbamazepina)',
        26 : 'Otras Sustancias'
    }

def UltimoMes(df):
    cols_a_eliminar = list(dict_impactosust().values()) + ['DrogaImpacto']
    df.drop(columns=[col for col in df.columns if col in cols_a_eliminar], inplace=True)
    list_UM = [col for col in df.columns if col.endswith('UltimoMes')]

    # Renombrar las columnas quitando el sufijo 'UltimoMes'
    new_names = {col: col.replace('UltimoMes', '') for col in list_UM}
    df.rename(columns=new_names, inplace=True)

    # Actualiza list_UM con los nuevos nombres
    list_UM = list(new_names.values())

    df['UltimoMes'] = df[list_UM].max(axis=1)
    df = df[df['UltimoMes'] > 0]
    df.drop(columns='UltimoMes', inplace=True)
    return df

def AlgunaVez(df):
    cols_a_eliminar= [col for col in df.columns if col.endswith('UltimoMes')]
    df = df.drop(columns=cols_a_eliminar + ['DrogaImpacto'])
    list_AV = list(dict_impactosust().values())
    df['AlgunaVez'] = df[list_AV].max(axis=1)
    df = df[df['AlgunaVez'] > 0]
    df.drop(columns='AlgunaVez', inplace=True)
    return df

def DrogaImpacto(df):
    UltimoMes = [col for col in df.columns if col.endswith('UltimoMes')]
    list_AV = list(dict_impactosust().values())
    df.drop(columns=UltimoMes + list_AV, inplace=True)
    dummies = pd.get_dummies(df['DrogaImpacto'])
    df = pd.concat([df, dummies], axis=1)
    df.drop(columns='DrogaImpacto', inplace=True)
    return df

def dict_grupos():
    return {
        'Tabaco' : 'Tabaco',
        'Alcohol' : 'Alcohol',
        'Marihuana' : 'Marihuana',
        'Hachis' : 'Marihuana',
        'Cocaina' : 'Cocaína',
        'Crack' : 'Cocaína',
        'Otras Presentaciones (Basuco o pasta base, cocaina negra)' : 'Cocaína',
        'Solventes y removedores' : 'Inhalables',
        'Pegamento' : 'Inhalables',
        'Esmaltes y pinturas' : 'Inhalables',
        'Otros (aire comprimido, gasolinas y combustibles)' : 'Inhalables',
        'Anfetaminas': 'Metanfetaminas',
        'Metanfetaminas' : 'Metanfetaminas',
        'MDMA(extasis) y metanfetaminas alucinogenas (DMT)' : 'Metanfetaminas',
        'Otros (derivados anfetaminicos)' : 'Metanfetaminas',
        'LSD' : 'Alucinógenos',
        'Plantas alucinogenas y derivados' : 'Alucinógenos',
        'Otras (PCP, ketamina, excepto metanfetamina)' : 'Alucinógenos',
        'Benzodiazepinas' : 'Medicamentos',
        'Rohypnol' : 'Medicamentos',
        'Otras (sedantes hiptnoticos, GHB)' : 'Medicamentos',
        'Heroina' : 'Opioides',
        'Opiaceos sinteticos (propoxifeno, nailbufina)' : 'Opioides',
        'Opio y opiodes (morfina, codeina)' : 'Opioides',
        'Con utilidad medica (Prozac, Paxil, Carbamazepina)' : 'Medicamentos',
        'Otras Sustancias' : 'Otras Sustancias'
    }

def DrogaImpactoGrupos(df):
    UltimoMes = [col for col in df.columns if col.endswith('UltimoMes')]
    list_AV = list(set(dict_grupos().keys())) 
    df.drop(columns=UltimoMes + list_AV, inplace=True)
    dummies = pd.get_dummies(df['DrogaImpacto'])
    df = pd.concat([df, dummies], axis=1)
    df.drop(columns='DrogaImpacto', inplace=True)
    return df

def Grupos(df):
    df['Marihuana'] = df[['Marihuana', 'Hachis']].max(axis=1)
    df['Cocaína'] = df[['Cocaina', 'Crack', 'Otras Presentaciones (Basuco o pasta base, cocaina negra)']].max(axis=1)
    df['Inhalables'] = df[['Solventes y removedores', 'Pegamento', 'Esmaltes y pinturas', 'Otros (aire comprimido, gasolinas y combustibles)']].max(axis=1)
    df['Metanfetaminas'] = df[['Anfetaminas', 'Metanfetaminas', 'MDMA(extasis) y metanfetaminas alucinogenas (DMT)', 'Otros (derivados anfetaminicos)']].max(axis=1)
    df['Alucinógenos'] = df[['LSD', 'Plantas alucinogenas y derivados', 'Otras (PCP, ketamina, excepto metanfetamina)']].max(axis=1)
    df['Medicamentos'] = df[['Benzodiazepinas', 'Rohypnol', 'Otras (sedantes hiptnoticos, GHB)', 'Con utilidad medica (Prozac, Paxil, Carbamazepina)']].max(axis=1)
    df['Opioides'] = df[['Heroina', 'Opiaceos sinteticos (propoxifeno, nailbufina)', 'Opio y opiodes (morfina, codeina)']].max(axis=1)
    cols_a_eliminar = [col for col in dict_impactosust().values() if col not in ['Tabaco', 'Alcohol', 'Metanfetaminas', 'Otras Sustancias', 'Marihuana']]
    df.drop(columns=cols_a_eliminar, inplace=True)
    return df

def AlgunaVezGrupos(df):
    colEliminar = [col for col in df.columns if col.endswith('UltimoMes')]
    df.drop(columns=colEliminar + ['DrogaImpacto'], inplace=True)
    list_AV = list(set(dict_grupos().values()))
    df['AlgunaVez'] = df[list_AV].max(axis=1)
    df = df[df['AlgunaVez'] > 0]
    df.drop(columns='AlgunaVez', inplace=True)
    return df

def UltimoMesGrupos(df):
    df.drop(columns=['DrogaImpacto', 'Tabaco', 'Alcohol', 'Marihuana', 'Cocaína', 'Inhalables', 'Metanfetaminas', 'Alucinógenos', 'Medicamentos', 'Opioides', 'Otras Sustancias'], inplace=True)
    df['Tabaco'] = df['TabacoUltimoMes']
    df['Alcohol'] = df['AlcoholUltimoMes']
    df['Marihuana'] = df[['MarihuanaUltimoMes', 'HachisUltimoMes']].max(axis=1)
    df['Cocaína'] = df[['CocainaUltimoMes', 'CrackUltimoMes', 'Otras Presentaciones (Basuco o pasta base, cocaina negra)UltimoMes']].max(axis=1)
    df['Inhalables'] = df[['Solventes y removedoresUltimoMes', 'PegamentoUltimoMes', 'Esmaltes y pinturasUltimoMes', 'Otros (aire comprimido, gasolinas y combustibles)UltimoMes']].max(axis=1)
    df['Metanfetaminas'] = df[['AnfetaminasUltimoMes', 'MetanfetaminasUltimoMes', 'MDMA(extasis) y metanfetaminas alucinogenas (DMT)UltimoMes', 'Otros (derivados anfetaminicos)UltimoMes']].max(axis=1)
    df['Alucinógenos'] = df[['LSDUltimoMes', 'Plantas alucinogenas y derivadosUltimoMes', 'Otras (PCP, ketamina, excepto metanfetamina)UltimoMes']].max(axis=1)
    df['Medicamentos'] = df[['BenzodiazepinasUltimoMes', 'RohypnolUltimoMes', 'Otras (sedantes hiptnoticos, GHB)UltimoMes', 'Con utilidad medica (Prozac, Paxil, Carbamazepina)UltimoMes']].max(axis=1)
    df['Opioides'] = df[['HeroinaUltimoMes', 'Opiaceos sinteticos (propoxifeno, nailbufina)UltimoMes', 'Opio y opiodes (morfina, codeina)UltimoMes']].max(axis=1)
    df['Otras Sustancias'] = df[['Otras SustanciasUltimoMes']].max(axis=1)
    df['UltimoMes'] = df[['Tabaco', 'Alcohol', 'Marihuana', 'Cocaína', 'Inhalables', 'Metanfetaminas', 'Alucinógenos', 'Medicamentos', 'Opioides', 'Otras Sustancias']].max(axis=1)
    df = df[df['UltimoMes'] > 0]
    colEliminar = [col for col in df.columns if col.endswith('UltimoMes')]
    df.drop(columns=colEliminar , inplace=True) 
    return df

In [3]:
def main():
    df = Read_data()
    df_UltimoMes = UltimoMes(df.copy())
    df_AlgunaVez = AlgunaVez(df.copy())
    df_DrogaImpacto = DrogaImpacto(df.copy())

    df['DrogaImpacto'] = df['DrogaImpacto'].map(dict_grupos())
    df_DrogaImpacto_Grupo = DrogaImpactoGrupos(df.copy())
    df_Grupo = Grupos(df.copy())
    df_AlgunaVez_Grupo = AlgunaVezGrupos(df_Grupo.copy())
    df_UltimoMes_Grupo = UltimoMesGrupos(df_Grupo.copy())

    listsust = list(dict_impactosust().values())
    for col in listsust:
        df_UltimoMes[col] = df_UltimoMes[col].fillna(0)
        df_AlgunaVez[col] = df_AlgunaVez[col].fillna(0)
        df_DrogaImpacto[col] = df_DrogaImpacto[col].fillna(0)
        df_UltimoMes[col] = df_UltimoMes[col].astype(bool)
        df_AlgunaVez[col] = df_AlgunaVez[col].astype(bool)
        df_DrogaImpacto[col] = df_DrogaImpacto[col].astype(bool)

    listgrupos = list(set(dict_grupos().values())) 
    for col in listgrupos:
        df_UltimoMes_Grupo[col] = df_UltimoMes_Grupo[col].fillna(0)
        df_AlgunaVez_Grupo[col] = df_AlgunaVez_Grupo[col].fillna(0)
        df_DrogaImpacto_Grupo[col] = df_DrogaImpacto_Grupo[col].fillna(0)
        df_UltimoMes_Grupo[col] = df_UltimoMes_Grupo[col].astype(bool)
        df_AlgunaVez_Grupo[col] = df_AlgunaVez_Grupo[col].astype(bool)
        df_DrogaImpacto_Grupo[col] = df_DrogaImpacto_Grupo[col].astype(bool)

    df_UltimoMes.to_csv('result_SetRiegs_Ilegales/ECERIECS(SI)_UltimoMes.csv', index=False)
    df_AlgunaVez.to_csv('result_SetRiegs_Ilegales/ECERIECS(SI)_AlgunaVez.csv', index=False)
    df_DrogaImpacto.to_csv('result_SetRiegs_Ilegales/ECERIECS(SI)_DrogaImpacto.csv', index=False)
    df_AlgunaVez_Grupo.to_csv('result_SetRiegs_Ilegales/ECERIECS(SI)_AlgunaVezGrupos.csv', index=False)
    df_UltimoMes_Grupo.to_csv('result_SetRiegs_Ilegales/ECERIECS(SI)_UltimoMesGrupos.csv', index=False)
    df_DrogaImpacto_Grupo.to_csv('result_SetRiegs_Ilegales/ECERIECS(SI)_DrogaImpactoGrupos.csv', index=False)
    

In [4]:
if __name__ == "__main__":
    main()